In [1]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error

from scipy.sparse import hstack
# Import the AutoTokenizer class from the transformers library
from transformers import AutoTokenizer,AutoModel

import os
import copy
import time

import random
from tqdm import tqdm
import torch.nn.functional as F
import torch
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, Dataset,SubsetRandomSampler
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
# from torchvision.models import resnet18, ResNet18_Weights
import torch
print(torch.cuda.is_available())
from torch.optim.lr_scheduler import ReduceLROnPlateau

from timm import create_model
import cv2

True


In [2]:
train = pd.read_csv('/kaggle/input/datasets/tridodai/essay-scoring/data/train.csv')
test  = pd.read_csv('/kaggle/input/datasets/tridodai/essay-scoring/data/test.csv')

print(train.head())

    id                                         essay_text  \
0   30  Climate change is one of the most urgent globa...   
1  536  Water pollution affects both human health and ...   
2  696  Education equips students with knowledge and c...   
3  558  Water pollution affects both human health and ...   
4  837  Reading daily improves vocabulary, comprehensi...   

                                     sentence_tokens  \
0  ['Climate change is one of the most urgent glo...   
1  ['Water pollution affects both human health an...   
2  ['Education equips students with knowledge and...   
3  ['Water pollution affects both human health an...   
4  ['Reading daily improves vocabulary, comprehen...   

                                         word_tokens  avg_sentence_length  \
0  ['Climate', 'change', 'is', 'one', 'of', 'the'...               10.000   
1  ['Water', 'pollution', 'affects', 'both', 'hum...                9.250   
2  ['Education', 'equips', 'students', 'with', 'k...             

In [3]:
TEXT_COL = 'essay_text'
TARGET = [
    'lexical_richness',
    'grammar_score',
    'coherence_score',
    'content_relevance_score',
    'final_score'
]


tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
# Làm sạch dữ liệu trước khi train
train = train.dropna(subset=[TEXT_COL] + TARGET).reset_index(drop=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
class EssayDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=128):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        encoding = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)

        return item

In [5]:
scaler = StandardScaler()

# Fit và transform các cột target
train_labels_scaled = scaler.fit_transform(train[TARGET].values)

# Khi split dữ liệu, sử dụng labels đã được chuẩn hóa
X_train, X_val, y_train, y_val = train_test_split(
    train[TEXT_COL].values,
    train_labels_scaled, # Sử dụng dữ liệu đã scale
    test_size=0.1,
    random_state=42
)

In [6]:
train_dataset = EssayDataset(X_train, y_train)
val_dataset   = EssayDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)

In [7]:
class BERTRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, 5)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        x = self.dropout(cls_output)
        out = self.fc(x)

        return out.squeeze()

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = BERTRegressor().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.1)
criterion = nn.MSELoss()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
from tqdm import tqdm
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

EPOCHS = 12

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    model.train()
    train_loss = 0
    train_preds = []
    train_targets = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")

    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)   # (batch, 5)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

        train_preds.extend(outputs.detach().cpu().numpy())
        train_targets.extend(labels.cpu().numpy())

        loop.set_postfix(loss=loss.item())

    # ===== TRAIN METRICS =====
    train_rmse = np.sqrt(mean_squared_error(train_targets, train_preds))
    train_mae  = mean_absolute_error(train_targets, train_preds)

    # ===== VALIDATION =====
    model.eval()
    val_preds = []
    val_targets = []

    loop = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]")

    with torch.no_grad():
        for batch in loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)

            val_preds.extend(outputs.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    # ===== VAL METRICS =====
    val_rmse = np.sqrt(mean_squared_error(val_targets, val_preds))
    val_mae  = mean_absolute_error(val_targets, val_preds)

    # ===== PRINT =====
    print(f"\nEpoch {epoch+1}:")
    print(f"Train RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f}")
    print(f"Val   RMSE: {val_rmse:.4f} | MAE: {val_mae:.4f}")

Epoch 1 [Val]: 100%|██████████| 3/3 [00:00<00:00,  5.54it/s]



Epoch 1:
Train RMSE: 0.9565 | MAE: 0.7861
Val   RMSE: 0.9198 | MAE: 0.7300


Epoch 2 [Val]: 100%|██████████| 3/3 [00:00<00:00,  5.38it/s]



Epoch 2:
Train RMSE: 0.9093 | MAE: 0.7285
Val   RMSE: 0.9154 | MAE: 0.7240


Epoch 3 [Val]: 100%|██████████| 3/3 [00:00<00:00,  5.24it/s]


...


Epoch 8:
Train RMSE: 0.8887 | MAE: 0.7048
Val   RMSE: 0.9233 | MAE: 0.7246


Epoch 9 [Val]: 100%|██████████| 3/3 [00:00<00:00,  4.77it/s]



Epoch 9:
Train RMSE: 0.8811 | MAE: 0.6966
Val   RMSE: 0.9414 | MAE: 0.7351


Epoch 10 [Val]: 100%|██████████| 3/3 [00:00<00:00,  4.65it/s]


Epoch 10:
Train RMSE: 0.8732 | MAE: 0.6933
Val   RMSE: 0.9463 | MAE: 0.7504


In [10]:
test_dataset = EssayDataset(test[TEXT_COL].values)
test_loader  = DataLoader(test_dataset, batch_size=8)

model.eval()
test_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, attention_mask)   # (batch, 5)
        test_preds.extend(outputs.cpu().numpy())

# 👉 convert sang numpy array
test_preds = np.array(test_preds)

test_preds_original_scale = scaler.inverse_transform(test_preds)
TARGET_COLS = [
    'lexical_richness',
    'grammar_score',
    'coherence_score',
    'content_relevance_score',
    'final_score'
]
# 👉 tạo submission với giá trị đã descale
submission = pd.DataFrame(test_preds_original_scale, columns=TARGET_COLS)
submission.insert(0, 'id', test['id'])

submission.to_csv('submission.csv', index=False)
